# **EXPLORATORY DATA ANALYSIS**

In [1]:
import pandas as pd

# **WEATHER**

In [2]:
df_weather = pd.read_parquet(r"C:\Users\Asus\Desktop\Formula1\data\bronze\weather\all_seasons_weather.parquet")
df_weather.head()

,season,round_number,race_name,circuit_ref,session_time_ms,air_temp_c,track_temp_c,humidity_pct,pressure_mbar,wind_speed_ms,wind_direction_deg,rainfall,is_wet_session,avg_track_temp,avg_air_temp,max_track_temp,min_track_temp
0,2018,1,Australian Grand Prix,Melbourne,49141.0,24.7,32.0,58.8,1007.9,1.3,307,False,False,30.47,24.74,32.6,28.8
1,2018,1,Australian Grand Prix,Melbourne,109222.0,24.7,31.9,58.8,1007.9,1.9,305,False,False,30.47,24.74,32.6,28.8
2,2018,1,Australian Grand Prix,Melbourne,169270.0,24.8,31.9,58.9,1007.9,2.1,279,False,False,30.47,24.74,32.6,28.8
3,2018,1,Australian Grand Prix,Melbourne,229300.0,24.7,31.9,58.9,1007.9,0.8,297,False,False,30.47,24.74,32.6,28.8
4,2018,1,Australian Grand Prix,Melbourne,289303.0,24.6,31.6,59.3,1007.9,1.5,293,False,False,30.47,24.74,32.6,28.8


In [3]:
df_weather.shape

(9002, 17)

- deriving aggregated columns

In [4]:
df_weather.drop(columns=["session_time_ms", "air_temp_c", "track_temp_c"], inplace=True)

In [5]:
df_weather.columns

Index(['season', 'round_number', 'race_name', 'circuit_ref', 'humidity_pct',
       'pressure_mbar', 'wind_speed_ms', 'wind_direction_deg', 'rainfall',
       'is_wet_session', 'avg_track_temp', 'avg_air_temp', 'max_track_temp',
       'min_track_temp'],
      dtype='object')

In [6]:
weather_summary = (
    df_weather
    .groupby(["season", "round_number", "race_name", "circuit_ref"], as_index=False)
    .agg(

        rainfall=("rainfall", "max"),
        is_wet_session=("is_wet_session", "max"),

        avg_humidity_pct=("humidity_pct", "mean"),
        avg_pressure_mbar=("pressure_mbar", "mean"),
        avg_wind_speed_ms=("wind_speed_ms", "mean"),
        avg_wind_direction_deg=("wind_direction_deg", "mean"),

        avg_track_temp=("avg_track_temp", "max"),
        avg_air_temp=("avg_air_temp", "max"),

        max_track_temp=("max_track_temp", "max"),
        min_track_temp=("min_track_temp", "min")
    )
)

In [7]:
weather_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   season                  107 non-null    int64  
 1   round_number            107 non-null    int64  
 2   race_name               107 non-null    object 
 3   circuit_ref             107 non-null    object 
 4   rainfall                107 non-null    bool   
 5   is_wet_session          107 non-null    bool   
 6   avg_humidity_pct        107 non-null    float64
 7   avg_pressure_mbar       107 non-null    float64
 8   avg_wind_speed_ms       107 non-null    float64
 9   avg_wind_direction_deg  107 non-null    float64
 10  avg_track_temp          107 non-null    float64
 11  avg_air_temp            107 non-null    float64
 12  max_track_temp          107 non-null    float64
 13  min_track_temp          107 non-null    float64
dtypes: bool(2), float64(8), int64(2), object(2

- no **PRIMARY KEY**
- creating **PRIMARY KEY** 
- combination of season and round_number columns is unique

In [8]:
weather_summary["weather_ref"] = weather_summary["season"].astype(str).str[2:4] + "_" + weather_summary["round_number"].astype(str)

In [13]:
weather_summary["avg_humidity_pct"] = round(weather_summary["avg_humidity_pct"],2)
weather_summary["avg_pressure_mbar"] = round(weather_summary["avg_pressure_mbar"],2)
weather_summary["avg_wind_speed_ms"] = round(weather_summary["avg_wind_speed_ms"],2)
weather_summary["avg_wind_direction_deg"] = round(weather_summary["avg_wind_direction_deg"],2)

In [14]:
weather_summary = weather_summary[["weather_ref", "season", "round_number", "race_name", "circuit_ref",
                           "rainfall", "is_wet_session", "avg_humidity_pct", "avg_pressure_mbar", "avg_wind_speed_ms", "avg_wind_direction_deg",
                           "avg_track_temp", "avg_air_temp", "max_track_temp", "min_track_temp"]]

In [19]:
weather_summary.head()

,weather_ref,season,round_number,race_name,circuit_ref,rainfall,is_wet_session,avg_humidity_pct,avg_pressure_mbar,avg_wind_speed_ms,avg_wind_direction_deg,avg_track_temp,avg_air_temp,max_track_temp,min_track_temp
0,18_1,2018,1,Australian Grand Prix,Melbourne,False,False,60.56,1007.90,0.78,216.67,30.47,24.74,32.6,28.8
1,18_2,2018,2,Bahrain Grand Prix,Sakhir,False,False,43.02,1008.69,0.72,145.78,31.86,28.21,32.5,31.4
2,18_3,2018,3,Chinese Grand Prix,Shanghai,False,False,76.15,1014.91,1.85,129.26,15.54,12.80,15.9,15.2
3,18_4,2018,4,Azerbaijan Grand Prix,Baku,True,True,46.21,1015.15,1.61,201.78,26.56,22.66,27.1,26.0
4,18_5,2018,5,Spanish Grand Prix,Barcelona,True,True,77.29,999.46,1.77,182.69,27.10,18.70,28.1,26.2


- dropping "is_wet_session" because "rainfall" and "is_wet_session" have same corresponding values 

In [20]:
weather_summary.drop(columns=["is_wet_session"], inplace=True)

In [21]:
len(weather_summary["weather_ref"].unique())

107

In [24]:
weather_summary.to_parquet(r"C:\Users\Asus\Desktop\Formula1\data\silver\weather\cleaned_weather.parquet")